In [ ]:
!pip install easyocr accelerate torch torchvision pandas opencv-python

In [ ]:
import os
import io
import glob
import json
import shutil
import zipfile
import requests
import pandas as pd
import cv2
import torch
import easyocr
import re
from transformers import AutoProcessor, AutoModelForImageTextToText, pipeline

In [ ]:
def init_models():
    print("1. Đang khởi tạo EasyOCR (tiếng Việt & tiếng Anh)...")
    reader = easyocr.Reader(['vi', 'en'], gpu=torch.cuda.is_available())
    
    print("2. Đang khởi tạo Qwen-Instruct để sửa lỗi chính tả OCR...")
    # Khởi tạo mô hình Qwen (yêu cầu GPU có VRAM khá)
    model_id = "Qwen/Qwen3-VL-2B-Instruct"
    
    processor = AutoProcessor.from_pretrained(model_id)
    qwen_model = AutoModelForImageTextToText.from_pretrained(
        model_id, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    
    return reader, processor, qwen_model

In [ ]:
def correct_text_with_qwen(raw_text: str, processor, model) -> str:
    """
    Sử dụng Qwen để đọc và sửa lỗi chính tả từ văn bản OCR.
    """
    if not raw_text.strip():
        return ""
        
    prompt = (
    "Bạn là mô hình hậu xử lý kết quả OCR tiếng Việt.\n\n"
    
    "Dưới đây là văn bản được trích xuất từ hình ảnh bằng OCR. "
    "Văn bản có thể chứa các lỗi như: sai chính tả, thiếu hoặc sai dấu tiếng Việt, "
    "nhầm ký tự, sai khoảng trắng, nối hoặc tách từ không đúng, "
    "và lỗi dấu câu do quá trình nhận dạng hình ảnh.\n\n"

    "NHIỆM VỤ:\n"
    "Chỉ sửa những lỗi OCR rõ ràng để văn bản chính xác và dễ đọc hơn. "
    "Phải giữ nguyên nội dung và ý nghĩa của văn bản gốc.\n\n"

    "QUY TẮC BẮT BUỘC:\n"
    "1. Không được thêm thông tin mới không xuất hiện trong văn bản gốc.\n"
    "2. Không được suy diễn nội dung dựa trên kiến thức bên ngoài.\n"
    "3. Không được tóm tắt, diễn giải hoặc viết lại câu.\n"
    "4. Không được tự ý thay đổi tên người, tên địa điểm, tên tổ chức, "
    "số điện thoại, ngày tháng, số liệu, mã số hoặc thuật ngữ chuyên ngành "
    "nếu không có lỗi OCR rõ ràng.\n"
    "5. Chỉ sửa lỗi khi có đủ cơ sở từ chính văn bản đầu vào.\n"
    "6. Nếu một từ hoặc cụm từ không chắc chắn có phải lỗi hay không, "
    "hãy giữ nguyên văn bản gốc.\n"
    "7. Giữ nguyên cấu trúc và thứ tự thông tin ban đầu càng nhiều càng tốt.\n"
    "8. Chỉ chuẩn hóa khoảng trắng và dấu câu khi cần thiết.\n"
    "9. Nếu văn bản đã đúng, trả lại nguyên văn không thay đổi.\n"
    "10. Chỉ trả về văn bản sau khi sửa, không giải thích và không thêm bất kỳ lời bình luận nào.\n\n"

    "VĂN BẢN OCR:\n"
    f"{raw_text}"
)
    
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    # Chuẩn bị input cho model
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text_input], return_tensors="pt").to(model.device)
    
    # Sinh văn bản (Inference)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        
    # Cắt bỏ phần prompt để chỉ lấy kết quả trả về
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    output_text = processor.batch_decode(
        generated_ids_trimmed, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )[0]
    
    return output_text.strip()

In [ ]:
def sort_ocr_boxes(boxes: list, texts: list, scores: list) -> tuple[list, list, list]:
    """
    Sắp xếp lại các box theo tọa độ (y trước, x sau)
    """
    # Ghép 3 list lại thành 1 list tuple để sort
    combined = list(zip(boxes, texts, scores))
    
    # Sort theo y_center (trung tâm chiều dọc) trước, x_center sau
    def get_y_center(item):
        box = item[0]
        return (box[1] + box[3]) / 2  # (y1 + y2)/2
    
    def get_x_center(item):
        box = item[0]
        return (box[0] + box[2]) / 2  # (x1 + x2)/2
    
    # Sắp xếp: ưu tiên y trước, nếu chênh lệch y nhỏ thì xếp theo x
    sorted_combined = sorted(combined, key=lambda item: (get_y_center(item), get_x_center(item)))
    
    # Tách ra lại
    sorted_boxes = [item[0] for item in sorted_combined]
    sorted_texts = [item[1] for item in sorted_combined]
    sorted_scores = [item[2] for item in sorted_combined]
    
    return sorted_boxes, sorted_texts, sorted_scores


def group_into_paragraphs(sorted_boxes: list, sorted_texts: list, line_height_threshold: float = 1.5) -> list[str]:
    """
    Gom các dòng OCR thành các đoạn văn hoàn chỉnh
    """
    if not sorted_boxes:
        return []
    
    # Tính chiều cao trung bình của các dòng
    heights = [box[3] - box[1] for box in sorted_boxes]
    avg_height = sum(heights) / len(heights)
    gap_threshold = avg_height * line_height_threshold
    
    paragraphs = []
    current_para = []
    prev_y2 = None
    
    for box, text in zip(sorted_boxes, sorted_texts):
        y1 = box[1]
        y2 = box[3]
        
        # Nếu là dòng đầu tiên hoặc khoảng cách với dòng trước <= ngưỡng -> gom chung
        if prev_y2 is None or (y1 - prev_y2) <= gap_threshold:
            current_para.append(text.strip())
        else:
            # Khoảng cách lớn -> kết thúc đoạn cũ, bắt đầu đoạn mới
            if current_para:
                paragraphs.append(" ".join(current_para))
            current_para = [text.strip()]
        
        prev_y2 = y2
    
    # Thêm đoạn cuối cùng
    if current_para:
        paragraphs.append(" ".join(current_para))
    
    return paragraphs


def clean_ocr_text(paragraphs: list[str]) -> list[str]:
    """
    Làm sạch từng đoạn văn bản OCR
    """
    cleaned_paragraphs = []
    for para in paragraphs:
        # 1. Nối từ bị gãy bởi dấu gạch nối cuối dòng (VD: "nguy- ên" -> "nguyên")
        para = re.sub(r'(\w+)\-\s+(\w+)', r'\1\2', para)
        
        # 2. Xóa các ký tự không phải chữ, số, dấu câu (giữ lại dấu . , ; : ! ?)
        para = re.sub(r'[^\w\s\.\,\;\:\!\?\(\)\"\'\-\+]', ' ', para, flags=re.UNICODE)
        
        # 3. Chuẩn hóa khoảng trắng (xóa dư thừa)
        para = re.sub(r'\s+', ' ', para).strip()
        
        cleaned_paragraphs.append(para)
    
    return cleaned_paragraphs

In [ ]:
def chunk_text_by_sentences(text: str, max_words: int = 400) -> list[str]:
    """
    Cắt đoạn văn thành các chunk nhỏ hơn dựa trên số từ
    """
    words = text.split()
    if len(words) <= max_words:
        return [text]
    
    chunks = []
    sentences = text.split('. ')
    temp = ""
    
    for sent in sentences:
        # Thêm dấu chấm nếu chưa có
        if not sent.endswith('.'):
            sent += '.'
        
        if len(temp.split()) + len(sent.split()) <= max_words:
            temp += sent + " "
        else:
            if temp:
                chunks.append(temp.strip())
            temp = sent + " "
    
    if temp:
        chunks.append(temp.strip())
    
    return chunks


def build_chunk_document(results_dict: dict, use_qwen_correction: bool = True) -> dict:
    """
    Xây dựng document object với các chunk text đã được xử lý
    """
    # B1: Sắp xếp boxes theo thứ tự đọc
    sorted_boxes, sorted_texts, sorted_scores = sort_ocr_boxes(
        results_dict["ocr_boxes"],
        results_dict["ocr_raw_texts"],
        results_dict["ocr_scores"]
    )
    
    # B2: Gom thành đoạn văn
    raw_paragraphs = group_into_paragraphs(sorted_boxes, sorted_texts)
    
    # B3: Làm sạch văn bản
    cleaned_paragraphs = clean_ocr_text(raw_paragraphs)
    
    # B4: Quyết định dùng text nào
    if use_qwen_correction and results_dict.get("qwen_corrected_text"):
        corrected_text = results_dict["qwen_corrected_text"]
        # Tách đoạn văn đã sửa thành các đoạn nhỏ (giả sử Qwen trả về có xuống dòng)
        corrected_paragraphs = [p.strip() for p in re.split(r'\n+', corrected_text) if p.strip()]
        final_texts = corrected_paragraphs if corrected_paragraphs else cleaned_paragraphs
    else:
        final_texts = cleaned_paragraphs
    
    # B5: Chunking - cắt đoạn quá dài
    chunked_texts = []
    for text in final_texts:
        if text.strip():
            chunks = chunk_text_by_sentences(text, max_words=400)
            chunked_texts.extend(chunks)
    
    # B6: Xây dựng document object
    document = {
        "metadata": {
            "total_boxes": len(sorted_boxes),
            "avg_ocr_score": sum(sorted_scores) / len(sorted_scores) if sorted_scores else 0,
            "has_qwen_correction": bool(results_dict.get("qwen_corrected_text"))
        },
        "chunks": []
    }
    
    for idx, chunk_text in enumerate(chunked_texts):
        chunk_object = {
            "chunk_id": idx,
            "text": chunk_text,
            "char_count": len(chunk_text),
            "word_count": len(chunk_text.split())
        }
        document["chunks"].append(chunk_object)
    
    # Lưu thêm thông tin raw để đối chiếu
    document["full_raw_text"] = results_dict["full_raw_text"]
    document["qwen_corrected_text"] = results_dict.get("qwen_corrected_text", "")
    
    return document


In [ ]:
def _extract_keyframe_info(img_path: str, fallback_index: int, video_id: str):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    numbers = re.findall(r'\d+', stem)

    if len(numbers) >= 2:
        keyframe_id = numbers[-2].zfill(4)
        frame_index = int(numbers[-1])
    elif len(numbers) == 1:
        keyframe_id = numbers[0].zfill(4)
        frame_index = int(fallback_index)
    else:
        keyframe_id = f"{fallback_index:04d}"
        frame_index = int(fallback_index)

    return str(video_id), keyframe_id, frame_index


def keyframes_to_ocr_json_per_frame(keyframe_paths: list[str], output_folder: str, models: tuple) -> None:
    reader, processor, qwen_model = models
    os.makedirs(output_folder, exist_ok=True)
    keyframe_paths = sorted(keyframe_paths)
    
    # Đã xóa dòng lấy video_id bằng output_folder ở đây

    for idx, img_path in enumerate(keyframe_paths, start=1):
        print(f"Processing frame {idx}: {img_path}")
        
        # 1. Trích xuất video_id động từ tên thư mục cha của bức ảnh (VD: V_76673ECA)
        current_video_id = os.path.basename(os.path.dirname(img_path))
        
        # 2. Tạo sub-folder tương ứng cho từng video bên trong thư mục output
        video_output_folder = os.path.join(output_folder, current_video_id)
        os.makedirs(video_output_folder, exist_ok=True)
        
        # Cập nhật truyền current_video_id vào hàm
        video_id, keyframe_id, frame_index = _extract_keyframe_info(img_path, idx, current_video_id)
        
        ocr_results = reader.readtext(img_path)
        texts = []
        for (bbox, text, prob) in ocr_results:
            xmin = int(min(pt[0] for pt in bbox))
            ymin = int(min(pt[1] for pt in bbox))
            xmax = int(max(pt[0] for pt in bbox))
            ymax = int(max(pt[1] for pt in bbox))
            texts.append({
                "text": str(text),
                "confidence": float(prob),
                "bbox": [xmin, ymin, xmax, ymax]
            })
            
        texts.sort(key=lambda item: (item["bbox"][1], item["bbox"][0]))
        
        results_dict = {
            "video_id": video_id,
            "keyframe_id": keyframe_id,
            "frame_index": int(frame_index),
            "texts": texts
        }
        
        file_name = f"{keyframe_id}.json"
        # 3. Trỏ đường dẫn lưu JSON vào đúng thư mục con của video đó
        output_json_path = os.path.join(video_output_folder, file_name)
        
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(results_dict, f, ensure_ascii=False, indent=2)
            
        print(f"  -> Saved: {output_json_path} with {len(texts)} text objects")

In [ ]:
def process_multiple_ocr_datasets(dataset_dirs: list, output_folder: str):
    """
    Quét đệ quy và xử lý toàn bộ ảnh từ danh sách các thư mục dataset bằng OCR.
    """
    image_paths = []
    # Khai báo các định dạng ảnh cần quét #
    image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG')
    
    print("1. Đang quét ảnh trong các thư mục dataset...")
    for ds_path in dataset_dirs:
        if not os.path.exists(ds_path):
            print(f" -> LỖI: Không tìm thấy thư mục {ds_path}")
            continue
            
        print(f" -> Đang quét: {ds_path}")
        for ext in image_extensions:
            # Quét đệ quy các thư mục con #
            image_paths.extend(glob.glob(os.path.join(ds_path, '**', ext), recursive=True))
            
    if not image_paths:
        print(" -> LỖI: Không tìm thấy ảnh nào trong các dataset được cung cấp!")
        return
        
    print(f" -> Tổng cộng tìm thấy {len(image_paths)} ảnh.")

    # Khởi tạo mô hình EasyOCR (và Qwen) #
    print("2. Đang khởi tạo các mô hình...")
    models = init_models()

    # Gọi hàm xử lý và xuất file JSON #[cite: 3]
    print(f"3. Bắt đầu chạy OCR và xuất kết quả JSON vào: {output_folder}")
    keyframes_to_ocr_json_per_frame(
        keyframe_paths=image_paths,
        output_folder=output_folder,
        models=models
    )
    
    print("\n=== HOÀN TẤT XỬ LÝ TOÀN BỘ DATASET ===")

def start_ocr_pipeline():
    print("Khởi động hệ thống OCR quét trực tiếp từ Kaggle Datasets...")
    
    # Danh sách 2 dataset bạn muốn quét
    dataset_dirs = [
        '/kaggle/input/datasets/keyframes'
    ]
    
    OUTPUT_JSON_DIR = '/kaggle/working/ocr_results'
    
    # Thực thi quá trình
    process_multiple_ocr_datasets(dataset_dirs, OUTPUT_JSON_DIR)

# Bắt đầu chạy pipeline
start_ocr_pipeline()